# Ari-LLM (Ari-LLM): High-Speed Cloud GPU Training (Kaggle / Colab)

Train our original decoder-only Transformer from scratch using a free NVIDIA T4 / P100 / A100 GPU on Kaggle.

### Why Kaggle GPU?
- 16 GB VRAM.
- ~20x faster training throughput compared to local CPU.
- Allows training a **12M - 43M** parameter model down to low cross-entropy loss (< 1.8) in just 15-25 minutes.

In [ ]:
# 1. Verify GPU Hardware and Clone Repository
!nvidia-smi
!git clone https://github.com/lucky4-git1/ARI.git
%cd ARI
!pip install -q -r requirements.txt

In [ ]:
# 2. Build Unified 12.4 MB Language & Code Corpus
!python scripts/download_corpus.py
!python scripts/build_unified_corpus.py
!python scripts/train_tokenizer_v2.py

In [ ]:
# 3. Run GPU Pretraining (1,500 - 3,000 steps)
# On Kaggle GPU, this processes millions of tokens in minutes with full CUDA acceleration.
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Training device: {device}')

# Run pretraining for 1,200 steps on GPU with batch_size 32 and seq_len 256
!python scripts/train.py \
    --corpus data/processed/combined_language_code_corpus.txt \
    --tokenizer artifacts/tokenizers/ari_tokenizer_v2.json \
    --max_steps 1500 \
    --batch_size 32 \
    --seq_len 256 \
    --hidden_dim 256 \
    --num_layers 8 \
    --num_heads 8 \
    --lr 0.0006 \
    --experiment_id exp_kaggle_gpu_v1

In [ ]:
# 4. Prepare Conversational SFT Dataset and Fine-Tune
!python scripts/prepare_sft_data.py
!python scripts/train_sft.py \
    --base_checkpoint artifacts/checkpoints/exp_kaggle_gpu_v1_best.pt \
    --tokenizer artifacts/tokenizers/ari_tokenizer_v2.json \
    --sft_data data/processed/sft_conversational_v2.jsonl \
    --max_steps 300 \
    --batch_size 8 \
    --lr 0.0002 \
    --output_checkpoint artifacts/checkpoints/exp_kaggle_sft_best.pt

In [ ]:
# 5. Test Inference in Notebook
!python scripts/generate.py \
    --checkpoint artifacts/checkpoints/exp_kaggle_sft_best.pt \
    --tokenizer artifacts/tokenizers/ari_tokenizer_v2.json \
    --prompt "<user>\nWhat is a Transformer model in artificial intelligence?\n<assistant>\n" \
    --max_tokens 60 \
    --temperature 0.2

In [ ]:
# 6. Package Trained Checkpoints for Download back to Local Machine
!zip -r ari_llm_trained_checkpoints.zip artifacts/checkpoints/exp_kaggle_sft_best.pt artifacts/tokenizers/ari_tokenizer_v2.json
print('Download ari_llm_trained_checkpoints.zip from the right-hand Kaggle Output pane!')